<a href="https://colab.research.google.com/github/sania934/EnergyMol-MLIP/blob/main/04_DFT_Dataset_Generation_with_PySCF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ase

In [ ]:
# Check whether PySCF is already installed.
# If not, install it.

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyscf") is None:
    print("PySCF not found. Installing...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pyscf"
    ])
    print("PySCF installation completed.")
else:
    print("PySCF is already installed.")

In [ ]:
# ============================================================
# CELL 04-INIT — RESTART-SAFE DFT ENVIRONMENT
# ============================================================
#
# Run this cell after every Colab runtime restart/reconnect.
#
# It:
#   1. Mounts Google Drive
#   2. Reconstructs project paths
#   3. Loads the CURRENT 563-configuration manifest
#   4. Reconstructs DFT directories
#   5. Checks PySCF + D3BJ
#   6. Restores run_pyscf_dft_job()
#
# It does NOT run the DFT dataset.
# It does NOT delete existing results.
# It does NOT rerun completed calculations.
# ============================================================

print("=" * 70)
print("ENERGYMOL-MLIP — RESTART-SAFE DFT INITIALIZATION")
print("=" * 70)


# ============================================================
# 1. GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

print("Google Drive mounted.")


# ============================================================
# 2. IMPORTS
# ============================================================

import sys
import json
import time
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

from ase.io import read
from ase.units import Hartree, Bohr

from pyscf import gto, dft
import pyscf


# ============================================================
# 3. PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MLIP project"
)

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

CONFIGS_DIR = PROJECT_DIR / "configs"

RESULTS_DIR = PROJECT_DIR / "results"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR = RESULTS_DIR / "figures"

DFT_DIR = PROJECT_DIR / "dft"

DFT_INPUT_DIR = DFT_DIR / "inputs"
DFT_OUTPUT_DIR = DFT_DIR / "outputs"
DFT_FAILED_DIR = DFT_DIR / "failed"
DFT_PARSED_DIR = DFT_DIR / "parsed"


# Create directories if needed

for directory in [
    DFT_DIR,
    DFT_INPUT_DIR,
    DFT_OUTPUT_DIR,
    DFT_FAILED_DIR,
    DFT_PARSED_DIR,
    RESULTS_DIR,
    METRICS_DIR,
    FIGURES_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print("\nProject directory:")
print(PROJECT_DIR)


# ============================================================
# 4. LOAD CURRENT CONFIGURATION MANIFEST
# ============================================================

MANIFEST_FILE = (
    PROCESSED_DIR /
    "configuration_manifest.csv"
)

if not MANIFEST_FILE.exists():

    raise FileNotFoundError(
        f"\nCurrent configuration manifest was not found:\n"
        f"{MANIFEST_FILE}\n\n"
        f"Please run Notebook 03 first."
    )


manifest = pd.read_csv(
    MANIFEST_FILE
)


# ============================================================
# 5. VALIDATE CURRENT DATASET
# ============================================================

print("\n" + "=" * 70)
print("CURRENT CONFIGURATION MANIFEST")
print("=" * 70)

print("Manifest :", MANIFEST_FILE)
print("Rows     :", len(manifest))
print("Columns  :", len(manifest.columns))


assert len(manifest) == 563, (
    f"Expected 563 configurations, "
    f"found {len(manifest)}"
)

assert manifest[
    "configuration_id"
].is_unique, (
    "configuration_id values are not unique."
)

assert manifest[
    "molecule_id"
].nunique() == 24, (
    "Expected 24 molecules."
)

assert manifest[
    "family"
].nunique() == 6, (
    "Expected 6 chemical families."
)


print("Configurations :", len(manifest))
print("Molecules      :", manifest["molecule_id"].nunique())
print("Families       :", manifest["family"].nunique())

print("\nCURRENT MANIFEST VALIDATED.")


# ============================================================
# 6. D3BJ BACKEND
# ============================================================

print("\n" + "=" * 70)
print("CHECKING PySCF / D3BJ")
print("=" * 70)

print("PySCF version:", pyscf.__version__)


# Important after Colab reconnect/restart

try:

    import pyscf.dispersion.dftd3

    import pyscf.scf.dispersion as scf_dispersion

    scf_dispersion = importlib.reload(
        scf_dispersion
    )

except Exception as exc:

    raise RuntimeError(
        "Could not load the PySCF D3 dispersion backend.\n"
        f"Error: {exc}"
    )


print(
    "D3BJ backend:",
    scf_dispersion.dftd3
)

print(
    "D3D4 backend:",
    scf_dispersion.dftd4
)

if scf_dispersion.dftd3 is None:

    raise RuntimeError(
        "\nD3BJ backend is unavailable.\n"
        "Run the pyscf-dispersion installation/recovery cell."
    )

print("\nD3BJ backend is AVAILABLE.")


# ============================================================
# 7. UNIT CONVERSIONS
# ============================================================

HARTREE_TO_EV = Hartree

HARTREE_BOHR_TO_EV_A = (
    Hartree / Bohr
)


# ============================================================
# 8. RESTORE DFT FUNCTION
# ============================================================

def run_pyscf_dft_job(
    xyz_file,
    output_file,
    method="PBE0",
    dispersion="D3BJ",
    basis="def2-SVP",
    charge=0,
    spin=0
):

    xyz_file = Path(xyz_file)
    output_file = Path(output_file)

    start_time = time.time()

    # --------------------------------------------------------
    # READ XYZ
    # --------------------------------------------------------

    atoms = read(
        str(xyz_file)
    )

    atom_data = [
        (
            symbol,
            tuple(
                float(x)
                for x in position
            )
        )
        for symbol, position in zip(
            atoms.get_chemical_symbols(),
            atoms.get_positions()
        )
    ]

    # --------------------------------------------------------
    # BUILD PySCF MOLECULE
    # --------------------------------------------------------

    mol = gto.M(
        atom=atom_data,
        basis=basis,
        charge=charge,
        spin=spin,
        unit="Angstrom",
        verbose=0
    )

    # --------------------------------------------------------
    # PBE0-D3(BJ)
    # --------------------------------------------------------

    mf = dft.RKS(mol)

    mf.xc = "pbe0-d3bj"

    mf.conv_tol = 1e-9
    mf.conv_tol_grad = 1e-6
    mf.max_cycle = 150

    # --------------------------------------------------------
    # SCF
    # --------------------------------------------------------

    energy_hartree = mf.kernel()

    if not mf.converged:

        raise RuntimeError(
            "SCF failed to converge."
        )

    # --------------------------------------------------------
    # ANALYTIC GRADIENT
    # --------------------------------------------------------

    grad_method = (
        mf.nuc_grad_method()
    )

    gradient_hartree_bohr = (
        grad_method.kernel()
    )

    # --------------------------------------------------------
    # VALIDATE GRADIENT
    # --------------------------------------------------------

    expected_shape = (
        mol.natm,
        3
    )

    if gradient_hartree_bohr.shape != expected_shape:

        raise RuntimeError(
            f"Unexpected gradient shape: "
            f"{gradient_hartree_bohr.shape}; "
            f"expected {expected_shape}"
        )

    if not np.all(
        np.isfinite(
            gradient_hartree_bohr
        )
    ):

        raise RuntimeError(
            "Gradient contains non-finite values."
        )

    # --------------------------------------------------------
    # ENERGY CONVERSION
    # --------------------------------------------------------

    energy_eV = (
        energy_hartree *
        HARTREE_TO_EV
    )

    # --------------------------------------------------------
    # FORCE CONVERSION
    # --------------------------------------------------------

    forces_eV_A = (
        -gradient_hartree_bohr *
        HARTREE_BOHR_TO_EV_A
    )

    # --------------------------------------------------------
    # RUNTIME
    # --------------------------------------------------------

    elapsed = (
        time.time() -
        start_time
    )

    # --------------------------------------------------------
    # RESULT
    # --------------------------------------------------------

    result = {

        "status": "success",

        "xyz_file":
            str(xyz_file),

        "n_atoms":
            int(mol.natm),

        "energy_hartree":
            float(energy_hartree),

        "energy_eV":
            float(energy_eV),

        "forces_eV_A":
            forces_eV_A.tolist(),

        "max_force_eV_A":
            float(
                np.max(
                    np.abs(
                        forces_eV_A
                    )
                )
            ),

        "scf_converged":
            bool(
                mf.converged
            ),

        "scf_cycles":
            int(
                getattr(
                    mf,
                    "cycles",
                    -1
                )
            ),

        "runtime_seconds":
            float(
                elapsed
            ),

        "method":
            method,

        "dispersion":
            dispersion,

        "basis":
            basis,

        "charge":
            charge,

        "spin":
            spin,

        "software":
            "PySCF"
    }

    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    output_file.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        output_file,
        "w"
    ) as f:

        json.dump(
            result,
            f,
            indent=2
        )

    return result


# ============================================================
# 9. FINAL ENVIRONMENT CHECK
# ============================================================

print("\n" + "=" * 70)
print("RESTART-SAFE ENVIRONMENT READY")
print("=" * 70)

print("Project       :", PROJECT_DIR)
print("Manifest rows :", len(manifest))
print("DFT output    :", DFT_OUTPUT_DIR)
print("PySCF         :", pyscf.__version__)
print("D3BJ          :", "AVAILABLE")
print(
    "DFT function  :",
    run_pyscf_dft_job
)

print("=" * 70)
print("READY FOR DFT BATCH")
print("=" * 70)

ENERGYMOL-MLIP — RESTART-SAFE DFT INITIALIZATION
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.

Project directory:
/content/drive/MyDrive/MLIP project

CURRENT CONFIGURATION MANIFEST
Manifest : /content/drive/MyDrive/MLIP project/data/processed/configuration_manifest.csv
Rows     : 563
Columns  : 15
Configurations : 563
Molecules      : 24
Families       : 6

CURRENT MANIFEST VALIDATED.

CHECKING PySCF / D3BJ
PySCF version: 2.14.0
D3BJ backend: <module 'pyscf.dispersion.dftd3' from '/usr/local/lib/python3.13/dist-packages/pyscf/dispersion/dftd3.py'>
D3D4 backend: <module 'pyscf.dispersion.dftd4' from '/usr/local/lib/python3.13/dist-packages/pyscf/dispersion/dftd4.py'>

D3BJ backend is AVAILABLE.

RESTART-SAFE ENVIRONMENT READY
Project       : /content/drive/MyDrive/MLIP project
Manifest rows : 563
DFT output    : /content/drive/MyDrive/MLIP project/dft/outputs
PySCF         : 2.14.0

In [ ]:
import os
import sys
import json
import time
import shutil
import traceback
import warnings

from pathlib import Path

import numpy as np
import pandas as pd

from ase import Atoms
from ase.io import read, write

import pyscf
from pyscf import gto, dft

warnings.filterwarnings("ignore")

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PySCF:", pyscf.__version__)

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
NumPy: 2.1.3
Pandas: 2.2.3
PySCF: 2.14.0


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/MLIP project")

# Existing directories
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

PILOT_STRUCTURES_DIR = PROJECT_DIR / "pilot_structures"
CONFIGS_DIR = PROJECT_DIR / "configs"

# New DFT directories
DFT_DIR = PROJECT_DIR / "dft"
DFT_INPUT_DIR = DFT_DIR / "inputs"
DFT_OUTPUT_DIR = DFT_DIR / "outputs"
DFT_FAILED_DIR = DFT_DIR / "failed"
DFT_PARSED_DIR = DFT_DIR / "parsed"

# Results
RESULTS_DIR = PROJECT_DIR / "results"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR = RESULTS_DIR / "figures"

for directory in [
    DFT_DIR,
    DFT_INPUT_DIR,
    DFT_OUTPUT_DIR,
    DFT_FAILED_DIR,
    DFT_PARSED_DIR,
    METRICS_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project directory:")
print(PROJECT_DIR)

print("\nNotebook 04 directories created.")
print("=" * 70)
print("PySCF ENVIRONMENT CHECK")
print("=" * 70)

print(f"PySCF version: {pyscf.__version__}")

try:
    import pyscf.grad
    print("Gradient module: available")
except Exception as e:
    print("Gradient module problem:", e)

print("\nEnvironment check completed.")

Project directory:
/content/drive/MyDrive/MLIP project

Notebook 04 directories created.
PySCF ENVIRONMENT CHECK
PySCF version: 2.14.0
Gradient module: available

Environment check completed.


In [ ]:
MANIFEST_FILE = PROCESSED_DIR / "configuration_manifest.csv"

if not MANIFEST_FILE.exists():
    raise FileNotFoundError(
        f"Configuration manifest not found:\n{MANIFEST_FILE}"
    )

config_df = pd.read_csv(MANIFEST_FILE)

print("=" * 70)
print("CONFIGURATION MANIFEST")
print("=" * 70)

print("Rows:", len(config_df))
print("Columns:")
print(config_df.columns.tolist())

display(config_df.head())

CONFIGURATION MANIFEST
Rows: 563
Columns:
['molecule_id', 'molecule_name', 'family', 'configuration_id', 'xyz_path', 'source_type', 'parent_conformer', 'perturbation_sigma_A', 'rmsd_from_parent_A', 'max_atom_displacement_A', 'mean_atom_displacement_A', 'n_atoms', 'n_heavy_atoms', 'finite_coordinates', 'minimum_distance_A']


,molecule_id,molecule_name,family,configuration_id,xyz_path,source_type,parent_conformer,perturbation_sigma_A,rmsd_from_parent_A,max_atom_displacement_A,mean_atom_displacement_A,n_atoms,n_heavy_atoms,finite_coordinates,minimum_distance_A
0,M01,benzene,aromatic_hydrocarbon,M01_C001,/content/drive/MyDrive/MLIP project/configs/M0...,mmff_conformer,0,0.000,1.510704e-16,0.000000,0.000000,12,6,True,1.086729
1,M01,benzene,aromatic_hydrocarbon,M01_C002,/content/drive/MyDrive/MLIP project/configs/M0...,perturbed,0,0.125,5.021607e-02,0.201182,0.117055,12,6,True,1.033968
2,M01,benzene,aromatic_hydrocarbon,M01_C003,/content/drive/MyDrive/MLIP project/configs/M0...,perturbed,0,0.125,5.010017e-02,0.287937,0.102942,12,6,True,1.074630
3,M01,benzene,aromatic_hydrocarbon,M01_C004,/content/drive/MyDrive/MLIP project/configs/M0...,perturbed,0,0.125,5.349264e-02,0.184807,0.111621,12,6,True,1.000970
4,M01,benzene,aromatic_hydrocarbon,M01_C005,/content/drive/MyDrive/MLIP project/configs/M0...,perturbed,0,0.125,5.143661e-02,0.215258,0.113172,12,6,True,1.021333


In [ ]:
required_columns = [
    "configuration_id",
    "molecule_id",
    "molecule_name",
]

missing_columns = [
    col for col in required_columns
    if col not in config_df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing from configuration manifest: "
        f"{missing_columns}"
    )

print("Required manifest columns found.")
def locate_xyz_file(row):
    """
    Locate the XYZ file associated with one configuration.
    """

    possible_paths = []

    # If manifest contains an explicit path
    for col in ["xyz_path", "configuration_file", "file_path"]:
        if col in row.index and pd.notna(row[col]):
            possible_paths.append(Path(str(row[col])))

    # Search under configs if explicit path is absent
    config_id = str(row["configuration_id"])
    molecule_id = str(row["molecule_id"])
    molecule_name = str(row["molecule_name"])

    possible_paths.extend([
        CONFIGS_DIR / molecule_id / f"{config_id}.xyz",
        CONFIGS_DIR / molecule_name / f"{config_id}.xyz",
    ])

    for path in possible_paths:
        if path.exists():
            return path

    # Recursive fallback search
    matches = list(CONFIGS_DIR.rglob(f"{config_id}.xyz"))

    if matches:
        return matches[0]

    return None

Required manifest columns found.


In [ ]:
xyz_paths = []

for _, row in config_df.iterrows():
    path = locate_xyz_file(row)
    xyz_paths.append(path)

config_df["xyz_path"] = [
    str(p) if p is not None else ""
    for p in xyz_paths
]

missing_xyz = config_df[
    config_df["xyz_path"].str.len() == 0
]

print("=" * 70)
print("XYZ FILE VALIDATION")
print("=" * 70)

print("Configurations:", len(config_df))
print("XYZ files found:", len(config_df) - len(missing_xyz))
print("XYZ files missing:", len(missing_xyz))

if len(missing_xyz) > 0:
    display(missing_xyz.head(20))
    raise FileNotFoundError(
        f"{len(missing_xyz)} configuration XYZ files could not be found."
    )

print("\nALL 576 XYZ FILES FOUND.")

XYZ FILE VALIDATION
Configurations: 563
XYZ files found: 563
XYZ files missing: 0

ALL 576 XYZ FILES FOUND.


In [ ]:
DFT_PROTOCOL = {
    "method": "PBE0",
    "dispersion": "D3BJ",
    "basis": "def2-SVP",
    "charge": 0,
    "multiplicity": 1,
    "spin": 0,
    "solvent": None,
    "calculation": "single_point_energy_gradient",
    "geometry_optimization": False,
    "temperature_K": None,
    "software": "PySCF",
}

print(json.dumps(DFT_PROTOCOL, indent=4))

{
    "method": "PBE0",
    "dispersion": "D3BJ",
    "basis": "def2-SVP",
    "charge": 0,
    "multiplicity": 1,
    "spin": 0,
    "solvent": null,
    "calculation": "single_point_energy_gradient",
    "geometry_optimization": false,
    "temperature_K": null,
    "software": "PySCF"
}


In [ ]:
PROTOCOL_FILE = CONFIGS_DIR / "dft_protocol.json"

with open(PROTOCOL_FILE, "w") as f:
    json.dump(DFT_PROTOCOL, f, indent=4)

print(f"\nDFT protocol saved to:\n{PROTOCOL_FILE}")


DFT protocol saved to:
/content/drive/MyDrive/MLIP project/configs/dft_protocol.json


In [ ]:
# ============================================================
# CELL 10 — PYSCF DISPERSION CAPABILITY DIAGNOSTIC
# ============================================================

import importlib.util
import pkgutil
import pyscf

print("=" * 70)
print("PYSCF DISPERSION CAPABILITY DIAGNOSTIC")
print("=" * 70)

print("PySCF version:", pyscf.__version__)
print("PySCF location:", pyscf.__file__)

print("\nRelevant installed PySCF modules:")

pyscf_modules = sorted(
    module.name
    for module in pkgutil.iter_modules(pyscf.__path__)
    if any(
        key in module.name.lower()
        for key in ["dft", "grad", "disp", "gto"]
    )
)

for module in pyscf_modules:
    print("  ", module)

print("\nChecking possible external dispersion packages:")

packages_to_check = [
    "dftd3",
    "pyscf_dftd3",
    "tblite",
    "geometric",
    "ase",
]

for package in packages_to_check:
    available = importlib.util.find_spec(package) is not None
    print(f"  {package:15s}: {'available' if available else 'not found'}")

print("\nDiagnostic completed.")

PYSCF DISPERSION CAPABILITY DIAGNOSTIC
PySCF version: 2.14.0
PySCF location: /usr/local/lib/python3.13/dist-packages/pyscf/__init__.py

Relevant installed PySCF modules:
   dft
   dispersion
   grad
   gto
   mcpdft
   tddft

Checking possible external dispersion packages:
  dftd3          : not found
  pyscf_dftd3    : not found
  tblite         : not found
  geometric      : not found
  ase            : available

Diagnostic completed.


In [ ]:
# ============================================================
# CELL 10B — CHECK PYSCF DFT SUBMODULES
# ============================================================

from pyscf import dft

print("=" * 70)
print("PYSCF.DFT MODULE CONTENTS")
print("=" * 70)

names = sorted(
    name for name in dir(dft)
    if not name.startswith("_")
)

for name in names:
    print(name)

PYSCF.DFT MODULE CONTENTS
BRAGG_RADII
COVALENT_RADII
DFT
DKS
GKS
Grids
KS
KohnShamDFT
LebedevGrid
RDKS
RKS
RKSpU
ROKS
UDKS
UKS
UKSpU
X2C
X2C_KS
XC
becke_atomic_radii_adjust
delley
density_fit
dks
gauss_chebyshev
gen_grid
gks
gks_symm
grid
gto
libxc
mura_knowles
numint
numint2c
nwchem_prune
original_becke
r_numint
radi
rks
rks_symm
roks
sg1_prune
stratmann
treutler
treutler_ahlrichs
treutler_atomic_radii_adjust
treutler_prune
uks
uks_symm
xc
xc_deriv
xcfun


In [ ]:
# ============================================================
# CELL 10C — INSTALL PYSCF DISPERSION EXTENSION
# ============================================================

import subprocess
import sys

print("=" * 70)
print("INSTALLING PYSCF DISPERSION EXTENSION")
print("=" * 70)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "pyscf-dispersion"
])

print("\nInstallation completed.")

INSTALLING PYSCF DISPERSION EXTENSION

Installation completed.


In [ ]:
# ============================================================
# CELL 10D — VERIFY D3 SUPPORT
# ============================================================

print("=" * 70)
print("VERIFYING PYSCF D3 SUPPORT")
print("=" * 70)

from pyscf import dispersion

print("PySCF dispersion module:", dispersion)

try:
    from pyscf.dispersion import dftd3
    print("DFTD3 backend: available")
except Exception as e:
    print("DFTD3 backend import failed:")
    print(repr(e))
    raise

print("\nD3 support is available.")

VERIFYING PYSCF D3 SUPPORT
PySCF dispersion module: <module 'pyscf.dispersion' from '/usr/local/lib/python3.13/dist-packages/pyscf/dispersion/__init__.py'>
DFTD3 backend: available

D3 support is available.


In [ ]:
# ============================================================
# CELL 10 — PYSCF DISPERSION CAPABILITY DIAGNOSTIC
# ============================================================

import importlib.util
import pkgutil
import pyscf

print("=" * 70)
print("PYSCF DISPERSION CAPABILITY DIAGNOSTIC")
print("=" * 70)

print("PySCF version:", pyscf.__version__)
print("PySCF location:", pyscf.__file__)

print("\nRelevant installed PySCF modules:")

pyscf_modules = sorted(
    module.name
    for module in pkgutil.iter_modules(pyscf.__path__)
    if any(
        key in module.name.lower()
        for key in ["dft", "grad", "disp", "gto"]
    )
)

for module in pyscf_modules:
    print("  ", module)

print("\nChecking possible external dispersion packages:")

packages_to_check = [
    "dftd3",
    "pyscf_dftd3",
    "tblite",
    "geometric",
    "ase",
]

for package in packages_to_check:
    available = importlib.util.find_spec(package) is not None
    print(f"  {package:15s}: {'available' if available else 'not found'}")

print("\nDiagnostic completed.")

PYSCF DISPERSION CAPABILITY DIAGNOSTIC
PySCF version: 2.14.0
PySCF location: /usr/local/lib/python3.13/dist-packages/pyscf/__init__.py

Relevant installed PySCF modules:
   dft
   dispersion
   grad
   gto
   mcpdft
   tddft

Checking possible external dispersion packages:
  dftd3          : not found
  pyscf_dftd3    : not found
  tblite         : not found
  geometric      : not found
  ase            : available

Diagnostic completed.


In [ ]:
# ============================================================
# CELL 10C — INSTALL PYSCF DISPERSION EXTENSION
# ============================================================

import subprocess
import sys

print("=" * 70)
print("INSTALLING PYSCF DISPERSION EXTENSION")
print("=" * 70)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "pyscf-dispersion"
])

print("\nInstallation completed.")

INSTALLING PYSCF DISPERSION EXTENSION

Installation completed.


In [ ]:
# ============================================================
# CELL 10E — TEST PBE0-D3(BJ) PARSING
# ============================================================

from pyscf import scf

print("=" * 70)
print("PBE0-D3(BJ) PARSING TEST")
print("=" * 70)

xc_test = "pbe0-d3bj"

xc_base, nlc, disp_version = scf.dispersion.parse_dft(xc_test)

print("Input functional :", xc_test)
print("Base XC           :", xc_base)
print("NLC               :", nlc)
print("Dispersion        :", disp_version)

if xc_base.lower() != "pbe0":
    raise ValueError(
        f"Unexpected base functional: {xc_base}"
    )

if disp_version.lower() != "d3bj":
    raise ValueError(
        f"Unexpected dispersion method: {disp_version}"
    )

print("\nPBE0-D3(BJ) was recognized correctly.")

PBE0-D3(BJ) PARSING TEST
Input functional : pbe0-d3bj
Base XC           : pbe0
NLC               : 
Dispersion        : d3bj

PBE0-D3(BJ) was recognized correctly.


In [ ]:
from pyscf import scf

print("D3 backend loaded successfully.")
print("pyscf.scf.dispersion module:", scf.dispersion)

D3 backend loaded successfully.
pyscf.scf.dispersion module: <module 'pyscf.scf.dispersion' from '/usr/local/lib/python3.13/dist-packages/pyscf/scf/dispersion.py'>


In [ ]:
# ============================================================
# CELL 11 — PYSCF TEST SETUP
# ============================================================

import time
import numpy as np

from pathlib import Path

from ase import Atoms
from ase.io import read

from pyscf import gto, dft

print("=" * 70)
print("PYSCF TEST SETUP")
print("=" * 70)


def ase_to_pyscf_mol(
    atoms,
    charge=0,
    spin=0,
    basis="def2-SVP"
):
    """
    Convert an ASE Atoms object into a PySCF Mole object.

    Parameters
    ----------
    atoms : ASE Atoms
        Molecular structure with coordinates in Å.
    charge : int
        Molecular charge.
    spin : int
        N_alpha - N_beta.
        spin=0 corresponds to a closed-shell system.
    basis : str
        PySCF basis-set name.

    Returns
    -------
    pyscf.gto.Mole
    """

    atom_data = [
        (
            symbol,
            tuple(float(x) for x in position)
        )
        for symbol, position in zip(
            atoms.get_chemical_symbols(),
            atoms.get_positions()
        )
    ]

    mol = gto.M(
        atom=atom_data,
        basis=basis,
        charge=charge,
        spin=spin,
        unit="Angstrom",
        verbose=4
    )

    return mol


# ------------------------------------------------------------
# Select test configuration
# ------------------------------------------------------------

TEST_CONFIG_ID = "M01_C001"

test_row = config_df[
    config_df["configuration_id"] == TEST_CONFIG_ID
]

if len(test_row) != 1:
    raise ValueError(
        f"Expected exactly one row for {TEST_CONFIG_ID}, "
        f"found {len(test_row)}"
    )

test_row = test_row.iloc[0]

TEST_XYZ = Path(test_row["xyz_path"])

if not TEST_XYZ.exists():
    raise FileNotFoundError(
        f"XYZ file not found:\n{TEST_XYZ}"
    )


# ------------------------------------------------------------
# Read structure
# ------------------------------------------------------------

test_atoms = read(TEST_XYZ)

print("Configuration ID :", TEST_CONFIG_ID)
print("Molecule          :", test_row["molecule_name"])
print("Family            :", test_row["family"])
print("Formula            :", test_atoms.get_chemical_formula())
print("Number of atoms   :", len(test_atoms))
print("XYZ file          :", TEST_XYZ)


# ------------------------------------------------------------
# Build PySCF molecule
# ------------------------------------------------------------

test_mol = ase_to_pyscf_mol(
    test_atoms,
    charge=DFT_PROTOCOL["charge"],
    spin=DFT_PROTOCOL["spin"],
    basis=DFT_PROTOCOL["basis"]
)

print("\nPySCF molecule created successfully.")
print("PySCF atoms       :", test_mol.natm)
print("Electrons         :", test_mol.nelectron)
print("Charge            :", test_mol.charge)
print("Spin              :", test_mol.spin)
print("Basis             :", DFT_PROTOCOL["basis"])

print("\nCell 11 completed successfully.")

PYSCF TEST SETUP
Configuration ID : M01_C001
Molecule          : benzene
Family            : aromatic_hydrocarbon
Formula            : C6H6
Number of atoms   : 12
XYZ file          : /content/drive/MyDrive/MLIP project/configs/M01_benzene/M01_C001.xyz
System: uname_result(system='Linux', node='486c8463a0d5', release='6.6.122+', version='#1 SMP Thu Apr 30 18:17:14 UTC 2026', machine='x86_64')  Threads 2
Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
numpy 2.1.3  scipy 1.16.3  h5py 3.16.0
Date: Sun Sep 27 12:55:31 2026
PySCF version 2.14.0
PySCF path  /usr/local/lib/python3.13/dist-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 12
[INPUT] num. electrons = 42
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = Angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 C     -0.8875285

In [ ]:
import sys

!{sys.executable} -m pip install pyscf-dispersion

In [ ]:
# ============================================================
# DIAGNOSTIC — WHY CAN PYSCF NOT LOAD DFTD3?
# ============================================================

import sys
import importlib.util
import pyscf
import pyscf.scf.dispersion as scf_dispersion

print("=" * 70)
print("PYSCF / DFTD3 DIAGNOSTIC")
print("=" * 70)

print("Python      :", sys.version)
print("PySCF       :", pyscf.__version__)
print("pyscf-dispersion module path:")
print(importlib.util.find_spec("pyscf.dispersion"))

print()
print("dftd3 module spec:")
print(importlib.util.find_spec("pyscf.dispersion.dftd3"))

print()
print("dftd4 module spec:")
print(importlib.util.find_spec("pyscf.dispersion.dftd4"))

print()
print("Inside pyscf.scf.dispersion:")
print("dftd3 =", scf_dispersion.dftd3)
print("dftd4 =", scf_dispersion.dftd4)

print()
print("Trying direct import of dftd3...")

try:
    from pyscf.dispersion import dftd3
    print("SUCCESS")
    print("dftd3 =", dftd3)
except Exception as e:
    print("FAILED")
    print(type(e).__name__ + ":", str(e))

PYSCF / DFTD3 DIAGNOSTIC
Python      : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PySCF       : 2.14.0
pyscf-dispersion module path:
ModuleSpec(name='pyscf.dispersion', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7cceff3e86b0>, origin='/usr/local/lib/python3.13/dist-packages/pyscf/dispersion/__init__.py', submodule_search_locations=['/usr/local/lib/python3.13/dist-packages/pyscf/dispersion'])

dftd3 module spec:
ModuleSpec(name='pyscf.dispersion.dftd3', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7cceed5453d0>, origin='/usr/local/lib/python3.13/dist-packages/pyscf/dispersion/dftd3.py')

dftd4 module spec:
ModuleSpec(name='pyscf.dispersion.dftd4', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7cceed314530>, origin='/usr/local/lib/python3.13/dist-packages/pyscf/dispersion/dftd4.py')

Inside pyscf.scf.dispersion:
dftd3 = None
dftd4 = None

Trying direct import of dftd3...
SUCCESS
dftd3 = <module 'pyscf.dispersion.dftd3' fro

In [ ]:
# ============================================================
# FIX: RELOAD PYSCF DISPERSION BACKEND
# ============================================================

import importlib
import pyscf.dispersion.dftd3

# Reload the SCF dispersion module after D3 is available
import pyscf.scf.dispersion as scf_dispersion
scf_dispersion = importlib.reload(scf_dispersion)

print("=" * 70)
print("D3 BACKEND RELOAD TEST")
print("=" * 70)

print("dftd3 =", scf_dispersion.dftd3)
print("dftd4 =", scf_dispersion.dftd4)

if scf_dispersion.dftd3 is None:
    raise RuntimeError("D3 backend is STILL unavailable.")

print()
print("SUCCESS: PySCF SCF module can now access DFTD3.")

D3 BACKEND RELOAD TEST
dftd3 = <module 'pyscf.dispersion.dftd3' from '/usr/local/lib/python3.13/dist-packages/pyscf/dispersion/dftd3.py'>
dftd4 = <module 'pyscf.dispersion.dftd4' from '/usr/local/lib/python3.13/dist-packages/pyscf/dispersion/dftd4.py'>

SUCCESS: PySCF SCF module can now access DFTD3.


In [ ]:
# ============================================================
# CELL 12 — PBE0-D3(BJ)/def2-SVP SCF TEST
# ============================================================

import time
import numpy as np
from pyscf import dft

print("=" * 70)
print("PBE0-D3(BJ)/def2-SVP SCF TEST")
print("=" * 70)

print("Functional :", "pbe0-d3bj")
print("Basis      :", test_mol.basis)
print("Charge     :", test_mol.charge)
print("Spin       :", test_mol.spin)

# ------------------------------------------------------------
# Build RKS calculation
# ------------------------------------------------------------

mf_test = dft.RKS(test_mol)
mf_test.xc = "pbe0-d3bj"

mf_test.conv_tol = 1e-9
mf_test.conv_tol_grad = 1e-6
mf_test.max_cycle = 150

# ------------------------------------------------------------
# Run SCF
# ------------------------------------------------------------

start_time = time.time()

energy_pbe0_d3bj_hartree = mf_test.kernel()

elapsed = time.time() - start_time

print()
print("=" * 70)
print("SCF RESULT")
print("=" * 70)

print("Converged :", mf_test.converged)
print(f"Energy    : {energy_pbe0_d3bj_hartree:.12f} Hartree")
print(f"Time      : {elapsed:.2f} seconds")

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

if not mf_test.converged:
    raise RuntimeError("PBE0-D3(BJ) SCF did not converge.")

if not np.isfinite(energy_pbe0_d3bj_hartree):
    raise RuntimeError("PBE0-D3(BJ) energy is not finite.")

print()
print("SCF ENERGY TEST PASSED.")

PBE0-D3(BJ)/def2-SVP SCF TEST
Functional : pbe0-d3bj


NameError: name 'test_mol' is not defined

In [ ]:
# ============================================================
# CELL 14 — PBE0-D3(BJ) NUCLEAR GRADIENT TEST
# ============================================================

import numpy as np

print("=" * 70)
print("PBE0-D3(BJ)/def2-SVP GRADIENT TEST")
print("=" * 70)

# Obtain nuclear gradient
grad_method = mf_test.nuc_grad_method()

total_gradient_hartree_bohr = grad_method.kernel()

print()
print("Gradient shape:", total_gradient_hartree_bohr.shape)
print("Expected shape:", (test_mol.natm, 3))

print()
print("Gradient statistics")
print("-" * 70)
print(
    f"Minimum : {np.min(total_gradient_hartree_bohr): .8e} Hartree/Bohr"
)
print(
    f"Maximum : {np.max(total_gradient_hartree_bohr): .8e} Hartree/Bohr"
)
print(
    f"Max abs : {np.max(np.abs(total_gradient_hartree_bohr)): .8e} Hartree/Bohr"
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

if total_gradient_hartree_bohr.shape != (test_mol.natm, 3):
    raise RuntimeError(
        f"Unexpected gradient shape: "
        f"{total_gradient_hartree_bohr.shape}"
    )

if not np.all(np.isfinite(total_gradient_hartree_bohr)):
    raise RuntimeError(
        "Gradient contains NaN or infinite values."
    )

print()
print("GRADIENT CALCULATION PASSED.")

PBE0-D3(BJ)/def2-SVP GRADIENT TEST


NameError: name 'mf_test' is not defined

In [ ]:
# ============================================================
# CELL 15 — ENERGY AND FORCE UNIT CONVERSION
# ============================================================

import numpy as np
from ase.units import Hartree, Bohr

print("=" * 70)
print("DFT ENERGY / FORCE UNIT CONVERSION")
print("=" * 70)

# ------------------------------------------------------------
# Energy
# Hartree -> eV
# ------------------------------------------------------------

energy_eV = energy_pbe0_d3bj_hartree * Hartree

# ------------------------------------------------------------
# Gradient
# Hartree/Bohr -> eV/Angstrom
# ------------------------------------------------------------

gradient_to_eV_A = Hartree / Bohr

gradient_eV_A = total_gradient_hartree_bohr * gradient_to_eV_A

# Forces are negative gradient
forces_eV_A = -gradient_eV_A

# ------------------------------------------------------------
# Print
# ------------------------------------------------------------

print(f"Energy       : {energy_eV:.10f} eV")
print(
    f"Force range  : "
    f"{forces_eV_A.min():.8f} to {forces_eV_A.max():.8f} eV/Angstrom"
)
print(
    f"Max |force|  : "
    f"{np.max(np.abs(forces_eV_A)):.8f} eV/Angstrom"
)

print()
print("Force array shape:", forces_eV_A.shape)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

if forces_eV_A.shape != (test_mol.natm, 3):
    raise RuntimeError(
        f"Unexpected force shape: {forces_eV_A.shape}"
    )

if not np.all(np.isfinite(forces_eV_A)):
    raise RuntimeError(
        "Forces contain NaN or infinite values."
    )

print()
print("ENERGY/FORCE UNIT CONVERSION PASSED.")

DFT ENERGY / FORCE UNIT CONVERSION
Energy       : -6307.9071641120 eV
Force range  : -0.22000099 to 0.21997740 eV/Angstrom
Max |force|  : 0.22000099 eV/Angstrom

Force array shape: (12, 3)

ENERGY/FORCE UNIT CONVERSION PASSED.


In [ ]:
# ============================================================
# CELL 16 — SELECT 12-CONFIGURATION DFT PILOT
# ============================================================

import pandas as pd
from pathlib import Path

print("=" * 70)
print("DFT PILOT SELECTION")
print("=" * 70)

# ------------------------------------------------------------
# Load configuration manifest
# ------------------------------------------------------------

manifest = pd.read_csv(MANIFEST_FILE)

print("Manifest rows:", len(manifest))
print("Configuration ID example:")
print(manifest["configuration_id"].head().tolist())

# ------------------------------------------------------------
# Representative molecules
# ------------------------------------------------------------
#
# M01 : benzene
# M09 : 1,2,3-thiadiazole
# M16 : 2-(4-nitrophenyl)thiophene
# M20 : 2,3,5,6-tetrafluoropyridine
#
# Three configurations per molecule:
# C001 = near-parent
# C012 = intermediate
# C024 = stronger perturbation
# ------------------------------------------------------------

pilot_molecules = {
    "M01": "benzene",
    "M09": "1,2,3-thiadiazole",
    "M16": "2-(4-nitrophenyl)thiophene",
    "M20": "2,3,5,6-tetrafluoropyridine",
}

pilot_config_suffixes = [
    "C001",
    "C012",
    "C024",
]

# ------------------------------------------------------------
# Select configurations
# ------------------------------------------------------------

pilot_rows = []

for molecule_id, molecule_name in pilot_molecules.items():

    # Configuration IDs are stored as M01_C001, etc.
    expected_ids = [
        f"{molecule_id}_{suffix}"
        for suffix in pilot_config_suffixes
    ]

    subset = manifest[
        manifest["configuration_id"].isin(expected_ids)
    ].copy()

    if len(subset) != 3:
        print()
        print("ERROR for:", molecule_id)
        print("Expected:", expected_ids)

        available = manifest[
            manifest["molecule_id"] == molecule_id
        ]["configuration_id"].tolist()

        print("Available:", available[:10])

        raise RuntimeError(
            f"Expected 3 configurations for {molecule_id}, "
            f"found {len(subset)}."
        )

    pilot_rows.append(subset)

# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

pilot_manifest = pd.concat(
    pilot_rows,
    ignore_index=True
)

# ------------------------------------------------------------
# Sort reproducibly
# ------------------------------------------------------------

pilot_manifest = pilot_manifest.sort_values(
    ["molecule_id", "configuration_id"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

if len(pilot_manifest) != 12:
    raise RuntimeError(
        f"Expected 12 pilot configurations, "
        f"found {len(pilot_manifest)}."
    )

if pilot_manifest["molecule_id"].nunique() != 4:
    raise RuntimeError(
        "Pilot does not contain exactly 4 molecules."
    )

# Check exactly 3 configurations per molecule
counts = pilot_manifest.groupby(
    "molecule_id"
).size()

if not (counts == 3).all():
    raise RuntimeError(
        f"Incorrect configurations per molecule:\n{counts}"
    )

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print()
print("=" * 70)
print("SELECTED PILOT CONFIGURATIONS")
print("=" * 70)

display(
    pilot_manifest[
        [
            "molecule_id",
            "molecule_name",
            "family",
            "configuration_id",
            "perturbation_sigma_A",
            "rmsd_from_parent_A",
            "max_atom_displacement_A",
            "xyz_path",
        ]
    ]
)

print()
print("=" * 70)
print("PILOT SUMMARY")
print("=" * 70)

print("Number of molecules :", pilot_manifest["molecule_id"].nunique())
print("Number of jobs      :", len(pilot_manifest))

print()
print("Configurations per molecule:")
print(counts)

print()
print("PILOT SELECTION PASSED.")

DFT PILOT SELECTION
Manifest rows: 563
Configuration ID example:
['M01_C001', 'M01_C002', 'M01_C003', 'M01_C004', 'M01_C005']

ERROR for: M01
Expected: ['M01_C001', 'M01_C012', 'M01_C024']
Available: ['M01_C001', 'M01_C002', 'M01_C003', 'M01_C004', 'M01_C005', 'M01_C006', 'M01_C007', 'M01_C008', 'M01_C009', 'M01_C010']


RuntimeError: Expected 3 configurations for M01, found 2.

In [ ]:
# ============================================================
# CELL 17 — CREATE DFT PILOT JOB DIRECTORIES
# ============================================================

from pathlib import Path
import json

print("=" * 70)
print("CREATING DFT PILOT JOB DIRECTORIES")
print("=" * 70)

# ------------------------------------------------------------
# Directory structure
# ------------------------------------------------------------

DFT_DIR.mkdir(parents=True, exist_ok=True)
DFT_INPUT_DIR.mkdir(parents=True, exist_ok=True)
DFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DFT_FAILED_DIR.mkdir(parents=True, exist_ok=True)
DFT_PARSED_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Create one directory per configuration
# ------------------------------------------------------------

for _, row in pilot_manifest.iterrows():

    job_id = row["configuration_id"]
    molecule_id = row["molecule_id"]

    job_dir = DFT_INPUT_DIR / f"{molecule_id}_{job_id}"
    job_dir.mkdir(parents=True, exist_ok=True)

print(f"Created job directories: {len(pilot_manifest)}")

# ------------------------------------------------------------
# Save pilot manifest
# ------------------------------------------------------------

PILOT_MANIFEST_FILE = (
    PROCESSED_DIR / "dft_pilot_manifest.csv"
)

pilot_manifest.to_csv(
    PILOT_MANIFEST_FILE,
    index=False
)

print()
print("Pilot manifest:")
print(PILOT_MANIFEST_FILE)

print()
print("DFT PILOT DIRECTORIES READY.")

CREATING DFT PILOT JOB DIRECTORIES


NameError: name 'pilot_manifest' is not defined

In [ ]:
# ============================================================
# CELL 18 — RESTARTABLE PYSCF DFT LABELING FUNCTION
# ============================================================

import json
import time
import traceback
import numpy as np
from pathlib import Path

from ase.io import read
from ase.units import Hartree, Bohr

from pyscf import gto, dft

# ------------------------------------------------------------
# Unit conversion
# ------------------------------------------------------------

HARTREE_TO_EV = Hartree
HARTREE_BOHR_TO_EV_A = Hartree / Bohr

# ------------------------------------------------------------
# DFT calculation function
# ------------------------------------------------------------

def run_pyscf_dft_job(
    xyz_file,
    output_file,
    method="PBE0",
    dispersion="D3BJ",
    basis="def2-SVP",
    charge=0,
    spin=0,
):

    xyz_file = Path(xyz_file)
    output_file = Path(output_file)

    start_time = time.time()

    # --------------------------------------------------------
    # Read XYZ
    # --------------------------------------------------------

    atoms = read(str(xyz_file))

    atom_data = [
        (
            symbol,
            tuple(float(x) for x in position)
        )
        for symbol, position in zip(
            atoms.get_chemical_symbols(),
            atoms.get_positions()
        )
    ]

    # --------------------------------------------------------
    # Build PySCF molecule
    # --------------------------------------------------------

    mol = gto.M(
        atom=atom_data,
        basis=basis,
        charge=charge,
        spin=spin,
        unit="Angstrom",
        verbose=0,
    )

    # --------------------------------------------------------
    # RKS / PBE0-D3(BJ)
    # --------------------------------------------------------

    mf = dft.RKS(mol)

    mf.xc = "pbe0-d3bj"

    mf.conv_tol = 1e-9
    mf.conv_tol_grad = 1e-6
    mf.max_cycle = 150

    # --------------------------------------------------------
    # SCF
    # --------------------------------------------------------

    energy_hartree = mf.kernel()

    if not mf.converged:
        raise RuntimeError(
            "SCF failed to converge."
        )

    # --------------------------------------------------------
    # Nuclear gradient
    # --------------------------------------------------------

    grad_method = mf.nuc_grad_method()

    gradient_hartree_bohr = grad_method.kernel()

    if gradient_hartree_bohr.shape != (mol.natm, 3):
        raise RuntimeError(
            f"Unexpected gradient shape: "
            f"{gradient_hartree_bohr.shape}"
        )

    if not np.all(np.isfinite(gradient_hartree_bohr)):
        raise RuntimeError(
            "Gradient contains NaN or infinite values."
        )

    # --------------------------------------------------------
    # Convert units
    # --------------------------------------------------------

    energy_eV = energy_hartree * HARTREE_TO_EV

    # Force = -gradient
    forces_eV_A = (
        -gradient_hartree_bohr
        * HARTREE_BOHR_TO_EV_A
    )

    # --------------------------------------------------------
    # Runtime
    # --------------------------------------------------------

    elapsed = time.time() - start_time

    # --------------------------------------------------------
    # Result dictionary
    # --------------------------------------------------------

    result = {
        "status": "success",
        "xyz_file": str(xyz_file),

        "n_atoms": int(mol.natm),

        "energy_hartree": float(energy_hartree),
        "energy_eV": float(energy_eV),

        "forces_eV_A": forces_eV_A.tolist(),

        "max_force_eV_A": float(
            np.max(np.abs(forces_eV_A))
        ),

        "scf_converged": bool(mf.converged),

        "scf_cycles": int(
            getattr(mf, "cycles", -1)
        ),

        "runtime_seconds": float(elapsed),

        "method": method,
        "dispersion": dispersion,
        "basis": basis,
        "charge": charge,
        "spin": spin,
        "software": "PySCF",
    }

    # --------------------------------------------------------
    # Save JSON
    # --------------------------------------------------------

    output_file.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(output_file, "w") as f:
        json.dump(
            result,
            f,
            indent=2
        )

    return result

In [ ]:
# ============================================================
# CELL 19 — RUN FIRST DFT PILOT JOB
# ============================================================

first_job = pilot_manifest.iloc[0]

job_id = (
    f"{first_job['molecule_id']}_"
    f"{first_job['configuration_id']}"
)

output_file = (
    DFT_OUTPUT_DIR /
    f"{job_id}.json"
)

print("=" * 70)
print("RUNNING FIRST DFT PILOT JOB")
print("=" * 70)

print("Job ID :", job_id)
print("Input  :", first_job["xyz_path"])
print("Output :", output_file)

result = run_pyscf_dft_job(
    xyz_file=first_job["xyz_path"],
    output_file=output_file,
)

print()
print("=" * 70)
print("RESULT")
print("=" * 70)

print("Status        :", result["status"])
print(
    f"Energy        : "
    f"{result['energy_eV']:.8f} eV"
)
print(
    f"Max |force|   : "
    f"{result['max_force_eV_A']:.8f} eV/Å"
)
print("SCF converged :", result["scf_converged"])
print("SCF cycles    :", result["scf_cycles"])
print(
    f"Runtime       : "
    f"{result['runtime_seconds']:.2f} s"
)

print()
print("FIRST DFT JOB COMPLETED.")

NameError: name 'pilot_manifest' is not defined

In [ ]:
# ============================================================
# CELL 19 — RUN AND VALIDATE FIRST DFT PILOT JOB
# ============================================================

print("=" * 70)
print("RUNNING FIRST DFT PILOT JOB")
print("=" * 70)

first_job = pilot_manifest.iloc[0]

# configuration_id already contains the molecule ID
# Example: M01_C001
job_id = first_job["configuration_id"]

input_file = Path(first_job["xyz_path"])
output_file = DFT_OUTPUT_DIR / f"{job_id}.json"

print(f"Job ID : {job_id}")
print(f"Input  : {input_file}")
print(f"Output : {output_file}")

# Run calculation
result = run_pyscf_dft_job(
    xyz_file=input_file,
    output_file=output_file,
    method="PBE0",
    dispersion="D3BJ",
    basis="def2-SVP",
    charge=0,
    spin=0
)

print("\n" + "=" * 70)
print("RESULT")
print("=" * 70)

print(f"Status        : {result['status']}")
print(f"Energy        : {result['energy_eV']:.8f} eV")
print(f"Max |force|   : {result['max_force_eV_A']:.8f} eV/Å")
print(f"SCF converged : {result['scf_converged']}")
print(f"SCF cycles    : {result['scf_cycles']}")
print(f"Runtime       : {result['runtime_seconds']:.2f} s")

# Basic validation
assert result["status"] == "success"
assert result["scf_converged"] is True
assert np.isfinite(result["energy_eV"])
assert np.isfinite(result["max_force_eV_A"])

forces = np.array(result["forces_eV_A"])

assert forces.shape == (
    result["n_atoms"],
    3
)

assert np.all(np.isfinite(forces))

print("\nFIRST DFT JOB COMPLETED AND VALIDATED.")

RUNNING FIRST DFT PILOT JOB


NameError: name 'pilot_manifest' is not defined

In [ ]:
# ============================================================
# CELL 20 — RUN DFT PILOT BATCH
# ============================================================

import time
import json
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("RUNNING DFT PILOT BATCH")
print("=" * 70)

pilot_results = []
successful = 0
failed = 0
skipped = 0

for i, row in pilot_manifest.iterrows():

    job_id = row["configuration_id"]

    input_file = Path(row["xyz_path"])
    output_file = DFT_OUTPUT_DIR / f"{job_id}.json"

    print("\n" + "-" * 70)
    print(f"[{i + 1}/{len(pilot_manifest)}] {job_id}")
    print(f"Molecule : {row['molecule_name']}")
    print(f"Family   : {row['family']}")
    print(f"Input    : {input_file}")
    print(f"Output   : {output_file}")

    # --------------------------------------------------------
    # Skip an already completed calculation
    # --------------------------------------------------------
    if output_file.exists():

        print("Status   : already completed — skipping")

        with open(output_file, "r") as f:
            result = json.load(f)

        result["configuration_id"] = job_id
        result["molecule_id"] = row["molecule_id"]
        result["molecule_name"] = row["molecule_name"]
        result["family"] = row["family"]

        pilot_results.append(result)
        skipped += 1

        continue

    # --------------------------------------------------------
    # Run DFT calculation
    # --------------------------------------------------------
    try:

        result = run_pyscf_dft_job(
            xyz_file=input_file,
            output_file=output_file,
            method="PBE0",
            dispersion="D3BJ",
            basis="def2-SVP",
            charge=0,
            spin=0
        )

        result["configuration_id"] = job_id
        result["molecule_id"] = row["molecule_id"]
        result["molecule_name"] = row["molecule_name"]
        result["family"] = row["family"]

        pilot_results.append(result)

        successful += 1

        print(f"Status        : {result['status']}")
        print(f"Energy        : {result['energy_eV']:.8f} eV")
        print(f"Max |force|   : {result['max_force_eV_A']:.8f} eV/Å")
        print(f"SCF converged : {result['scf_converged']}")
        print(f"SCF cycles    : {result['scf_cycles']}")
        print(f"Runtime       : {result['runtime_seconds']:.2f} s")

    except Exception as exc:

        failed += 1

        error_record = {
            "configuration_id": job_id,
            "molecule_id": row["molecule_id"],
            "molecule_name": row["molecule_name"],
            "family": row["family"],
            "status": "failed",
            "error": str(exc)
        }

        pilot_results.append(error_record)

        print("STATUS: FAILED")
        print(f"Error: {exc}")

        failed_file = DFT_FAILED_DIR / f"{job_id}_failed.json"

        with open(failed_file, "w") as f:
            json.dump(error_record, f, indent=2)


# ============================================================
# SAVE PILOT RESULTS
# ============================================================

pilot_results_df = pd.DataFrame(pilot_results)

pilot_summary_file = RESULTS_DIR / "dft_pilot_results.csv"

pilot_results_df.to_csv(
    pilot_summary_file,
    index=False
)

print("\n" + "=" * 70)
print("DFT PILOT BATCH COMPLETE")
print("=" * 70)

print(f"Total pilot jobs : {len(pilot_manifest)}")
print(f"New successful   : {successful}")
print(f"Skipped existing : {skipped}")
print(f"Failed           : {failed}")

print("\nSummary saved to:")
print(pilot_summary_file)

RUNNING DFT PILOT BATCH


NameError: name 'pilot_manifest' is not defined

In [ ]:
# ============================================================
# CELL 21 — INSPECT DFT PILOT RESULTS
# ============================================================

print("=" * 70)
print("DFT PILOT RESULTS")
print("=" * 70)

pilot_results_df = pd.read_csv(
    pilot_summary_file
)

display(
    pilot_results_df[
        [
            "configuration_id",
            "molecule_name",
            "family",
            "status",
            "energy_eV",
            "max_force_eV_A",
            "scf_converged",
            "scf_cycles",
            "runtime_seconds"
        ]
    ].sort_values(
        "configuration_id"
    )
)

DFT PILOT RESULTS


NameError: name 'pilot_summary_file' is not defined

In [ ]:
# ============================================================
# CELL 22 — VALIDATE DFT PILOT DATA
# ============================================================

print("=" * 70)
print("VALIDATING DFT PILOT DATA")
print("=" * 70)

required_columns = [
    "configuration_id",
    "molecule_id",
    "molecule_name",
    "family",
    "status",
    "energy_eV",
    "max_force_eV_A",
    "scf_converged",
    "scf_cycles",
    "runtime_seconds"
]

missing_columns = [
    col for col in required_columns
    if col not in pilot_results_df.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

success_df = pilot_results_df[
    pilot_results_df["status"] == "success"
].copy()

failed_df = pilot_results_df[
    pilot_results_df["status"] != "success"
].copy()

print(f"Total jobs      : {len(pilot_results_df)}")
print(f"Successful jobs : {len(success_df)}")
print(f"Failed jobs     : {len(failed_df)}")

if len(failed_df) > 0:
    print("\nFailed jobs:")
    display(failed_df)

assert len(success_df) == 12, (
    f"Expected 12 successful pilot calculations, "
    f"but found {len(success_df)}."
)

assert success_df["scf_converged"].all()

assert np.all(
    np.isfinite(success_df["energy_eV"].values)
)

assert np.all(
    np.isfinite(success_df["max_force_eV_A"].values)
)

assert np.all(
    success_df["runtime_seconds"].values > 0
)

print("\nALL 12 PILOT CALCULATIONS PASSED BASIC VALIDATION.")

VALIDATING DFT PILOT DATA


NameError: name 'pilot_results_df' is not defined

In [ ]:
# ============================================================
# CELL 23 — CHECK ENERGY, FORCE AND RUNTIME
# ============================================================

print("=" * 70)
print("PILOT ENERGY, FORCE AND RUNTIME RANGES")
print("=" * 70)

print(
    f"Energy minimum      : "
    f"{success_df['energy_eV'].min():.8f} eV"
)

print(
    f"Energy maximum      : "
    f"{success_df['energy_eV'].max():.8f} eV"
)

print(
    f"Maximum force min   : "
    f"{success_df['max_force_eV_A'].min():.8f} eV/Å"
)

print(
    f"Maximum force max   : "
    f"{success_df['max_force_eV_A'].max():.8f} eV/Å"
)

print(
    f"Mean runtime        : "
    f"{success_df['runtime_seconds'].mean():.2f} s"
)

print(
    f"Median runtime      : "
    f"{success_df['runtime_seconds'].median():.2f} s"
)

print("\nPilot results:")

display(
    success_df[
        [
            "configuration_id",
            "molecule_id",
            "molecule_name",
            "family",
            "energy_eV",
            "max_force_eV_A",
            "runtime_seconds"
        ]
    ].sort_values(
        ["molecule_id", "configuration_id"]
    )
)

PILOT ENERGY, FORCE AND RUNTIME RANGES


NameError: name 'success_df' is not defined

In [ ]:
# ============================================================
# CELL 24 — MERGE CONFIGURATION METADATA
# ============================================================

print("=" * 70)
print("MERGING CONFIGURATION METADATA")
print("=" * 70)

configuration_columns = [
    "configuration_id",
    "perturbation_sigma_A",
    "rmsd_from_parent_A",
    "max_atom_displacement_A",
    "mean_atom_displacement_A",
    "minimum_distance_A",
    "n_atoms",
    "n_heavy_atoms"
]

pilot_analysis_df = success_df.merge(
    pilot_manifest[configuration_columns],
    on="configuration_id",
    how="left"
)

assert len(pilot_analysis_df) == 12

metadata_columns = [
    "perturbation_sigma_A",
    "rmsd_from_parent_A",
    "max_atom_displacement_A",
    "mean_atom_displacement_A",
    "minimum_distance_A"
]

for column in metadata_columns:
    assert pilot_analysis_df[column].notna().all(), (
        f"Missing configuration metadata: {column}"
    )

display(
    pilot_analysis_df[
        [
            "configuration_id",
            "molecule_name",
            "family",
            "perturbation_sigma_A",
            "rmsd_from_parent_A",
            "max_atom_displacement_A",
            "energy_eV",
            "max_force_eV_A"
        ]
    ].sort_values(
        "configuration_id"
    )
)

MERGING CONFIGURATION METADATA


NameError: name 'success_df' is not defined

In [ ]:
# ============================================================
# CELL 25 — CALCULATE RELATIVE ENERGIES
# ============================================================

print("=" * 70)
print("CALCULATING RELATIVE ENERGIES")
print("=" * 70)

pilot_analysis_df["delta_E_eV"] = np.nan

for molecule_id in pilot_analysis_df["molecule_id"].unique():

    mask = (
        pilot_analysis_df["molecule_id"] == molecule_id
    )

    minimum_energy = pilot_analysis_df.loc[
        mask,
        "energy_eV"
    ].min()

    pilot_analysis_df.loc[
        mask,
        "delta_E_eV"
    ] = (
        pilot_analysis_df.loc[
            mask,
            "energy_eV"
        ]
        - minimum_energy
    )

assert pilot_analysis_df["delta_E_eV"].notna().all()

display(
    pilot_analysis_df[
        [
            "configuration_id",
            "molecule_name",
            "perturbation_sigma_A",
            "rmsd_from_parent_A",
            "energy_eV",
            "delta_E_eV",
            "max_force_eV_A"
        ]
    ].sort_values(
        "configuration_id"
    )
)

CALCULATING RELATIVE ENERGIES


NameError: name 'pilot_analysis_df' is not defined

In [ ]:
# ============================================================
# CELL 26 — CREATE DFT PILOT QC PLOTS
# ============================================================

import matplotlib.pyplot as plt

# ------------------------------------------------------------
# PLOT 1 — RELATIVE ENERGY VS RMSD
# ------------------------------------------------------------

plt.figure(figsize=(7, 5))

for molecule_id, group in pilot_analysis_df.groupby(
    "molecule_id"
):

    plt.scatter(
        group["rmsd_from_parent_A"],
        group["delta_E_eV"],
        label=group["molecule_name"].iloc[0]
    )

plt.xlabel("RMSD from parent (Å)")
plt.ylabel("Relative energy ΔE (eV)")
plt.title(
    "DFT Pilot: Relative Energy vs Structural Distortion"
)

plt.legend()
plt.tight_layout()

plot1 = (
    FIGURES_DIR /
    "dft_pilot_relative_energy_vs_rmsd.png"
)

plt.savefig(
    plot1,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Saved: {plot1}")


# ------------------------------------------------------------
# PLOT 2 — MAXIMUM FORCE VS RMSD
# ------------------------------------------------------------

plt.figure(figsize=(7, 5))

for molecule_id, group in pilot_analysis_df.groupby(
    "molecule_id"
):

    plt.scatter(
        group["rmsd_from_parent_A"],
        group["max_force_eV_A"],
        label=group["molecule_name"].iloc[0]
    )

plt.xlabel("RMSD from parent (Å)")
plt.ylabel("Maximum |force| (eV/Å)")
plt.title(
    "DFT Pilot: Maximum Force vs Structural Distortion"
)

plt.legend()
plt.tight_layout()

plot2 = (
    FIGURES_DIR /
    "dft_pilot_max_force_vs_rmsd.png"
)

plt.savefig(
    plot2,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Saved: {plot2}")

In [ ]:
# ============================================================
# CELL 27 — SAVE PILOT ANALYSIS
# ============================================================

pilot_analysis_file = (
    RESULTS_DIR /
    "dft_pilot_analysis.csv"
)

pilot_analysis_df.to_csv(
    pilot_analysis_file,
    index=False
)

print("=" * 70)
print("PILOT ANALYSIS SAVED")
print("=" * 70)

print(f"File : {pilot_analysis_file}")
print(f"Rows : {len(pilot_analysis_df)}")

print("\nPilot analysis completed successfully.")

In [ ]:
# ============================================================
# CELL 28 — PREPARE FULL DFT DATASET
# ============================================================

import json
import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 70)
print("PREPARING FULL DFT DATASET")
print("=" * 70)

# ------------------------------------------------------------
# Reload the FINAL 563-configuration manifest
# ------------------------------------------------------------

MANIFEST_FILE = (
    PROCESSED_DIR /
    "configuration_manifest.csv"
)

manifest = pd.read_csv(MANIFEST_FILE)

print(f"Configurations : {len(manifest)}")
print(f"Molecules      : {manifest['molecule_id'].nunique()}")
print(f"Families       : {manifest['family'].nunique()}")

assert len(manifest) == 563
assert manifest["configuration_id"].is_unique

# ------------------------------------------------------------
# Required directories
# ------------------------------------------------------------

DFT_DIR.mkdir(parents=True, exist_ok=True)
DFT_INPUT_DIR.mkdir(parents=True, exist_ok=True)
DFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DFT_FAILED_DIR.mkdir(parents=True, exist_ok=True)
DFT_PARSED_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Check all XYZ files
# ------------------------------------------------------------

missing_xyz = []

for _, row in manifest.iterrows():

    xyz_file = Path(row["xyz_path"])

    if not xyz_file.exists():
        missing_xyz.append(str(xyz_file))

print(f"Missing XYZ files : {len(missing_xyz)}")

assert len(missing_xyz) == 0, (
    f"{len(missing_xyz)} XYZ files are missing."
)

# ------------------------------------------------------------
# Count existing DFT results
# ------------------------------------------------------------

completed = 0
remaining = 0

for job_id in manifest["configuration_id"]:

    output_file = DFT_OUTPUT_DIR / f"{job_id}.json"

    if output_file.exists():
        completed += 1
    else:
        remaining += 1

print(f"Existing DFT results : {completed}")
print(f"Remaining DFT jobs   : {remaining}")

print("\nFULL DFT DATASET IS READY.")

PREPARING FULL DFT DATASET


NameError: name 'PROCESSED_DIR' is not defined

In [ ]:
# ============================================================
# CELL 29 — CREATE FINAL DFT JOB MANIFEST
# ============================================================

print("=" * 70)
print("CREATING FINAL DFT JOB MANIFEST")
print("=" * 70)

final_dft_manifest = manifest.copy()

final_dft_manifest["output_json"] = (
    final_dft_manifest["configuration_id"]
    .apply(
        lambda x: str(
            DFT_OUTPUT_DIR / f"{x}.json"
        )
    )
)

final_dft_manifest["status"] = (
    final_dft_manifest["configuration_id"]
    .apply(
        lambda x:
        "completed"
        if (
            DFT_OUTPUT_DIR / f"{x}.json"
        ).exists()
        else "pending"
    )
)

FINAL_DFT_MANIFEST_FILE = (
    PROCESSED_DIR /
    "dft_full_manifest.csv"
)

final_dft_manifest.to_csv(
    FINAL_DFT_MANIFEST_FILE,
    index=False
)

print(f"Manifest saved to:")
print(FINAL_DFT_MANIFEST_FILE)

print("\nStatus summary:")
print(
    final_dft_manifest["status"]
    .value_counts()
)

display(
    final_dft_manifest[
        [
            "molecule_id",
            "molecule_name",
            "family",
            "configuration_id",
            "status"
        ]
    ].head(20)
)

CREATING FINAL DFT JOB MANIFEST


NameError: name 'manifest' is not defined

In [ ]:
# ============================================================
# D3BJ BACKEND RECOVERY
# ============================================================

import sys
import subprocess
import importlib

print("=" * 70)
print("CHECKING PYS CF D3BJ BACKEND")
print("=" * 70)

# 1. Install/reinstall the required dispersion package
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade",
    "pyscf-dispersion"
])

print("\npyscf-dispersion installation completed.")

# 2. Reload PySCF dispersion modules
import pyscf
import pyscf.dispersion.dftd3
import pyscf.scf.dispersion as scf_dispersion

scf_dispersion = importlib.reload(scf_dispersion)

print("PySCF version:", pyscf.__version__)
print("dftd3 object :", scf_dispersion.dftd3)
print("dftd4 object :", scf_dispersion.dftd4)

assert scf_dispersion.dftd3 is not None, (
    "D3BJ backend is still unavailable."
)

print("\n" + "=" * 70)
print("D3BJ BACKEND IS AVAILABLE")
print("=" * 70)

CHECKING PYS CF D3BJ BACKEND

pyscf-dispersion installation completed.
PySCF version: 2.14.0
dftd3 object : <module 'pyscf.dispersion.dftd3' from '/usr/local/lib/python3.13/dist-packages/pyscf/dispersion/dftd3.py'>
dftd4 object : <module 'pyscf.dispersion.dftd4' from '/usr/local/lib/python3.13/dist-packages/pyscf/dispersion/dftd4.py'>

D3BJ BACKEND IS AVAILABLE


In [ ]:
# ============================================================
# TEST D3BJ ON ONE FAILED CONFIGURATION
# ============================================================

test_job = "M04_C016"

test_row = manifest[
    manifest["configuration_id"] == test_job
].iloc[0]

test_input = Path(test_row["xyz_path"])
test_output = DFT_OUTPUT_DIR / f"{test_job}_retest.json"

print("=" * 70)
print("D3BJ RETEST")
print("=" * 70)
print("Job      :", test_job)
print("Molecule :", test_row["molecule_name"])
print("Input    :", test_input)
print("Output   :", test_output)

result = run_pyscf_dft_job(
    xyz_file=test_input,
    output_file=test_output,
    method="PBE0",
    dispersion="D3BJ",
    basis="def2-SVP",
    charge=0,
    spin=0
)

print("\n" + "=" * 70)
print("RETEST SUCCESSFUL")
print("=" * 70)
print(f"Energy      : {result['energy_eV']:.8f} eV")
print(f"Max |force| : {result['max_force_eV_A']:.8f} eV/Å")
print(f"SCF cycles  : {result['scf_cycles']}")
print(f"Runtime     : {result['runtime_seconds']:.2f} s")

NameError: name 'manifest' is not defined

In [ ]:
# ============================================================
# CELL 30 — RUN FULL DFT DATASET
# ============================================================

import json
import time
import traceback
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("RUNNING FULL DFT DATASET")
print("=" * 70)

# ------------------------------------------------------------
# Define final manifest path directly
# ------------------------------------------------------------

FINAL_DFT_MANIFEST_FILE = (
    PROCESSED_DIR /
    "dft_full_manifest.csv"
)

# Check that the manifest exists
if not FINAL_DFT_MANIFEST_FILE.exists():
    raise FileNotFoundError(
        f"Final DFT manifest not found:\n"
        f"{FINAL_DFT_MANIFEST_FILE}\n\n"
        "Run Cell 29 first."
    )

# ------------------------------------------------------------
# Load final 563-configuration manifest
# ------------------------------------------------------------

manifest = pd.read_csv(
    FINAL_DFT_MANIFEST_FILE
)

print(f"Total configurations : {len(manifest)}")

assert len(manifest) == 563
assert manifest["configuration_id"].is_unique

# ------------------------------------------------------------
# Make sure DFT directories exist
# ------------------------------------------------------------

DFT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DFT_FAILED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Run DFT calculations
# ------------------------------------------------------------

successful = 0
skipped = 0
failed = 0

start_all = time.time()

for i, row in manifest.iterrows():

    job_id = row["configuration_id"]

    input_file = Path(row["xyz_path"])
    output_file = (
        DFT_OUTPUT_DIR /
        f"{job_id}.json"
    )
    failed_file = (
        DFT_FAILED_DIR /
        f"{job_id}_failed.json"
    )

    print("\n" + "=" * 70)
    print(
        f"JOB {i + 1}/{len(manifest)}"
    )
    print(f"ID       : {job_id}")
    print(f"Molecule : {row['molecule_name']}")
    print(f"Family   : {row['family']}")
    print("=" * 70)

    # --------------------------------------------------------
    # Skip completed jobs
    # --------------------------------------------------------

    if output_file.exists():

        print("STATUS: COMPLETED — SKIPPING")

        skipped += 1
        continue

    # --------------------------------------------------------
    # Check input XYZ
    # --------------------------------------------------------

    if not input_file.exists():

        print("STATUS: FAILED — XYZ FILE NOT FOUND")

        error_record = {
            "configuration_id": job_id,
            "molecule_id": row["molecule_id"],
            "molecule_name": row["molecule_name"],
            "family": row["family"],
            "status": "failed",
            "error": f"XYZ file not found: {input_file}"
        }

        with open(failed_file, "w") as f:
            json.dump(
                error_record,
                f,
                indent=2
            )

        failed += 1
        continue

    # --------------------------------------------------------
    # Run PySCF DFT
    # --------------------------------------------------------

    try:

        result = run_pyscf_dft_job(
            xyz_file=input_file,
            output_file=output_file,
            method="PBE0",
            dispersion="D3BJ",
            basis="def2-SVP",
            charge=0,
            spin=0
        )

        # Add configuration metadata
        result["configuration_id"] = job_id
        result["molecule_id"] = row["molecule_id"]
        result["molecule_name"] = row["molecule_name"]
        result["family"] = row["family"]
        result["perturbation_sigma_A"] = float(
            row["perturbation_sigma_A"]
        )
        result["rmsd_from_parent_A"] = float(
            row["rmsd_from_parent_A"]
        )

        # Save updated JSON
        with open(output_file, "w") as f:
            json.dump(
                result,
                f,
                indent=2
            )

        successful += 1

        print("STATUS: SUCCESS")
        print(
            f"Energy      : "
            f"{result['energy_eV']:.8f} eV"
        )
        print(
            f"Max |force| : "
            f"{result['max_force_eV_A']:.8f} eV/Å"
        )
        print(
            f"SCF cycles  : "
            f"{result['scf_cycles']}"
        )
        print(
            f"Runtime     : "
            f"{result['runtime_seconds']:.2f} s"
        )

    except Exception as exc:

        failed += 1

        error_record = {
            "configuration_id": job_id,
            "molecule_id": row["molecule_id"],
            "molecule_name": row["molecule_name"],
            "family": row["family"],
            "status": "failed",
            "error": str(exc),
            "traceback": traceback.format_exc()
        }

        with open(failed_file, "w") as f:
            json.dump(
                error_record,
                f,
                indent=2
            )

        print("STATUS: FAILED")
        print(f"Error: {exc}")


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

elapsed_all = time.time() - start_all

print("\n" + "=" * 70)
print("FULL DFT RUN FINISHED")
print("=" * 70)

print(f"Total jobs       : {len(manifest)}")
print(f"New successful   : {successful}")
print(f"Skipped existing : {skipped}")
print(f"Failed           : {failed}")
print(
    f"Elapsed time     : "
    f"{elapsed_all / 3600:.2f} hours"
)

print("=" * 70)

RUNNING FULL DFT DATASET
Total configurations : 563

JOB 1/563
ID       : M01_C001
Molecule : benzene
Family   : aromatic_hydrocarbon
STATUS: COMPLETED — SKIPPING

JOB 2/563
ID       : M01_C002
Molecule : benzene
Family   : aromatic_hydrocarbon
STATUS: COMPLETED — SKIPPING

JOB 3/563
ID       : M01_C003
Molecule : benzene
Family   : aromatic_hydrocarbon
STATUS: COMPLETED — SKIPPING

JOB 4/563
ID       : M01_C004
Molecule : benzene
Family   : aromatic_hydrocarbon
STATUS: COMPLETED — SKIPPING

JOB 5/563
ID       : M01_C005
Molecule : benzene
Family   : aromatic_hydrocarbon
STATUS: COMPLETED — SKIPPING

JOB 6/563
ID       : M01_C006
Molecule : benzene
Family   : aromatic_hydrocarbon
STATUS: COMPLETED — SKIPPING

JOB 7/563
ID       : M01_C007
Molecule : benzene
Family   : aromatic_hydrocarbon
STATUS: COMPLETED — SKIPPING

JOB 8/563
ID       : M01_C008
Molecule : benzene
Family   : aromatic_hydrocarbon
STATUS: COMPLETED — SKIPPING

JOB 9/563
ID       : M01_C009
Molecule : benzene
Family   :

KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 30B — RUN DFT IN BATCHES
# ============================================================

import json
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# BATCH SETTINGS
# ------------------------------------------------------------

BATCH_START = 87     # inclusive
BATCH_END   = 156    # inclusive

batch_manifest = manifest.iloc[BATCH_START - 1:BATCH_END].copy()

print("=" * 70)
print("DFT BATCH")
print("=" * 70)
print(f"Batch range : {BATCH_START} - {BATCH_END}")
print(f"Jobs in batch: {len(batch_manifest)}")
print("=" * 70)

successful = 0
skipped = 0
failed = 0

for batch_number, (_, row) in enumerate(
    batch_manifest.iterrows(),
    start=BATCH_START
):

    job_id = row["configuration_id"]

    input_file = Path(row["xyz_path"])
    output_file = DFT_OUTPUT_DIR / f"{job_id}.json"

    print("\n" + "-" * 70)
    print(f"JOB {batch_number}/563")
    print(f"ID       : {job_id}")
    print(f"Molecule : {row['molecule_name']}")
    print(f"Family   : {row['family']}")

    # --------------------------------------------------------
    # CHECK EXISTING OUTPUT
    # --------------------------------------------------------

    if output_file.exists():

        try:
            with open(output_file, "r") as f:
                old_result = json.load(f)

            forces = np.asarray(
                old_result.get("forces_eV_A", []),
                dtype=float
            )

            energy = old_result.get("energy_eV", None)

            valid_existing = (
                old_result.get("status") == "success"
                and energy is not None
                and np.isfinite(float(energy))
                and forces.shape == (int(row["n_atoms"]), 3)
                and np.all(np.isfinite(forces))
                and old_result.get("scf_converged", False)
            )

            if valid_existing:
                print("STATUS: COMPLETED — SKIPPING")
                skipped += 1
                continue

        except Exception:
            pass

    # --------------------------------------------------------
    # RUN DFT
    # --------------------------------------------------------

    print("STATUS: RUNNING")

    try:

        result = run_pyscf_dft_job(
            xyz_file=input_file,
            output_file=output_file,
            method="PBE0",
            dispersion="D3BJ",
            basis="def2-SVP",
            charge=0,
            spin=0
        )

        # Add current manifest metadata
        result.update({
            "configuration_id": job_id,
            "molecule_id": row["molecule_id"],
            "molecule_name": row["molecule_name"],
            "family": row["family"],
        })

        with open(output_file, "w") as f:
            json.dump(result, f, indent=2)

        successful += 1

        print(
            f"STATUS: SUCCESS | "
            f"E = {result['energy_eV']:.6f} eV | "
            f"max|F| = {result['max_force_eV_A']:.6f} eV/A"
        )

    except Exception as exc:

        failed += 1

        print("STATUS: FAILED")
        print("Error:", exc)

        failed_file = (
            DFT_FAILED_DIR /
            f"{job_id}_failed.json"
        )

        failure_record = {
            "configuration_id": job_id,
            "molecule_id": row["molecule_id"],
            "molecule_name": row["molecule_name"],
            "family": row["family"],
            "status": "failed",
            "error": str(exc),
            "xyz_file": str(input_file)
        }

        with open(failed_file, "w") as f:
            json.dump(failure_record, f, indent=2)

# ------------------------------------------------------------
# BATCH SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BATCH COMPLETE")
print("=" * 70)
print(f"Batch range : {BATCH_START} - {BATCH_END}")
print(f"Successful  : {successful}")
print(f"Skipped     : {skipped}")
print(f"Failed      : {failed}")
print("=" * 70)

DFT BATCH
Batch range : 87 - 156
Jobs in batch: 70

----------------------------------------------------------------------
JOB 87/563
ID       : M04_C016
Molecule : anthracene
Family   : aromatic_hydrocarbon
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------------
JOB 88/563
ID       : M04_C017
Molecule : anthracene
Family   : aromatic_hydrocarbon
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------------
JOB 89/563
ID       : M04_C018
Molecule : anthracene
Family   : aromatic_hydrocarbon
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------------
JOB 90/563
ID       : M04_C019
Molecule : anthracene
Family   : aromatic_hydrocarbon
STATUS: RUNNING
STATUS: SUCCESS | E = -14651.744384 eV | max|F| = 6.468731 eV/A

----------------------------------------------------------------------
JOB 91/563
ID       : M04_C020
Molecule : anthracene
Family   : aromatic_hydrocarb

In [ ]:
# ============================================================
# RESTORE run_pyscf_dft_job()
# PBE0-D3(BJ)/def2-SVP
# Single-point energy + analytic nuclear gradient
# ============================================================

import json
import time
import numpy as np
from pathlib import Path

from ase.io import read
from ase.units import Hartree, Bohr

from pyscf import gto, dft


# ------------------------------------------------------------
# UNIT CONVERSIONS
# ------------------------------------------------------------

HARTREE_TO_EV = Hartree
HARTREE_BOHR_TO_EV_A = Hartree / Bohr


# ------------------------------------------------------------
# DFT JOB FUNCTION
# ------------------------------------------------------------

def run_pyscf_dft_job(
    xyz_file,
    output_file,
    method="PBE0",
    dispersion="D3BJ",
    basis="def2-SVP",
    charge=0,
    spin=0
):

    xyz_file = Path(xyz_file)
    output_file = Path(output_file)

    start_time = time.time()

    # --------------------------------------------------------
    # READ XYZ
    # --------------------------------------------------------

    atoms = read(str(xyz_file))

    atom_data = [
        (
            symbol,
            tuple(float(x) for x in position)
        )
        for symbol, position
        in zip(
            atoms.get_chemical_symbols(),
            atoms.get_positions()
        )
    ]

    # --------------------------------------------------------
    # BUILD PySCF MOLECULE
    # --------------------------------------------------------

    mol = gto.M(
        atom=atom_data,
        basis=basis,
        charge=charge,
        spin=spin,
        unit="Angstrom",
        verbose=0
    )

    # --------------------------------------------------------
    # PBE0-D3(BJ)
    # --------------------------------------------------------

    mf = dft.RKS(mol)

    mf.xc = "pbe0-d3bj"

    mf.conv_tol = 1e-9
    mf.conv_tol_grad = 1e-6
    mf.max_cycle = 150

    # --------------------------------------------------------
    # SCF
    # --------------------------------------------------------

    energy_hartree = mf.kernel()

    if not mf.converged:
        raise RuntimeError(
            "SCF failed to converge."
        )

    # --------------------------------------------------------
    # ANALYTIC NUCLEAR GRADIENT
    # --------------------------------------------------------

    grad_method = mf.nuc_grad_method()

    gradient_hartree_bohr = grad_method.kernel()

    # --------------------------------------------------------
    # VALIDATE GRADIENT
    # --------------------------------------------------------

    expected_shape = (mol.natm, 3)

    if gradient_hartree_bohr.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected gradient shape: "
            f"{gradient_hartree_bohr.shape}; "
            f"expected {expected_shape}"
        )

    if not np.all(
        np.isfinite(gradient_hartree_bohr)
    ):
        raise RuntimeError(
            "Gradient contains non-finite values."
        )

    # --------------------------------------------------------
    # CONVERT ENERGY
    # --------------------------------------------------------

    energy_eV = (
        energy_hartree *
        HARTREE_TO_EV
    )

    # --------------------------------------------------------
    # GRADIENT -> FORCE
    #
    # Force = -gradient
    # --------------------------------------------------------

    forces_eV_A = (
        -gradient_hartree_bohr *
        HARTREE_BOHR_TO_EV_A
    )

    # --------------------------------------------------------
    # RUNTIME
    # --------------------------------------------------------

    elapsed = time.time() - start_time

    # --------------------------------------------------------
    # RESULT
    # --------------------------------------------------------

    result = {

        "status": "success",

        "xyz_file": str(xyz_file),

        "n_atoms": int(mol.natm),

        "energy_hartree":
            float(energy_hartree),

        "energy_eV":
            float(energy_eV),

        "forces_eV_A":
            forces_eV_A.tolist(),

        "max_force_eV_A":
            float(
                np.max(
                    np.abs(forces_eV_A)
                )
            ),

        "scf_converged":
            bool(mf.converged),

        "scf_cycles":
            int(
                getattr(
                    mf,
                    "cycles",
                    -1
                )
            ),

        "runtime_seconds":
            float(elapsed),

        "method":
            method,

        "dispersion":
            dispersion,

        "basis":
            basis,

        "charge":
            charge,

        "spin":
            spin,

        "software":
            "PySCF"
    }

    # --------------------------------------------------------
    # SAVE RESULT
    # --------------------------------------------------------

    output_file.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        output_file,
        "w"
    ) as f:

        json.dump(
            result,
            f,
            indent=2
        )

    return result


# ============================================================
# FUNCTION CHECK
# ============================================================

print("=" * 70)
print("DFT FUNCTION RESTORED")
print("=" * 70)

print(run_pyscf_dft_job)

print("\nPySCF version:", __import__("pyscf").__version__)

print("=" * 70)

DFT FUNCTION RESTORED
<function run_pyscf_dft_job at 0x78885637a520>

PySCF version: 2.14.0


In [ ]:
print(run_pyscf_dft_job)

<function run_pyscf_dft_job at 0x78885637a520>


In [ ]:
# ============================================================
# CELL 30C — RUN DFT IN BATCHES
# ============================================================

# ============================================================
# CELL 30B — RUN DFT IN BATCHES
# ============================================================

import json
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# BATCH SETTINGS
# ------------------------------------------------------------

BATCH_START = 157     # inclusive
BATCH_END   = 226     # inclusive

batch_manifest = manifest.iloc[BATCH_START - 1:BATCH_END].copy()

print("=" * 70)
print("DFT BATCH")
print("=" * 70)
print(f"Batch range : {BATCH_START} - {BATCH_END}")
print(f"Jobs in batch: {len(batch_manifest)}")
print("=" * 70)

successful = 0
skipped = 0
failed = 0

for batch_number, (_, row) in enumerate(
    batch_manifest.iterrows(),
    start=BATCH_START
):

    job_id = row["configuration_id"]

    input_file = Path(row["xyz_path"])
    output_file = DFT_OUTPUT_DIR / f"{job_id}.json"

    print("\n" + "-" * 70)
    print(f"JOB {batch_number}/563")
    print(f"ID       : {job_id}")
    print(f"Molecule : {row['molecule_name']}")
    print(f"Family   : {row['family']}")

    # --------------------------------------------------------
    # CHECK EXISTING OUTPUT
    # --------------------------------------------------------

    if output_file.exists():

        try:
            with open(output_file, "r") as f:
                old_result = json.load(f)

            forces = np.asarray(
                old_result.get("forces_eV_A", []),
                dtype=float
            )

            energy = old_result.get("energy_eV", None)

            valid_existing = (
                old_result.get("status") == "success"
                and energy is not None
                and np.isfinite(float(energy))
                and forces.shape == (int(row["n_atoms"]), 3)
                and np.all(np.isfinite(forces))
                and old_result.get("scf_converged", False)
            )

            if valid_existing:
                print("STATUS: COMPLETED — SKIPPING")
                skipped += 1
                continue

        except Exception:
            pass

    # --------------------------------------------------------
    # RUN DFT
    # --------------------------------------------------------

    print("STATUS: RUNNING")

    try:

        result = run_pyscf_dft_job(
            xyz_file=input_file,
            output_file=output_file,
            method="PBE0",
            dispersion="D3BJ",
            basis="def2-SVP",
            charge=0,
            spin=0
        )

        # Add current manifest metadata
        result.update({
            "configuration_id": job_id,
            "molecule_id": row["molecule_id"],
            "molecule_name": row["molecule_name"],
            "family": row["family"],
        })

        with open(output_file, "w") as f:
            json.dump(result, f, indent=2)

        successful += 1

        print(
            f"STATUS: SUCCESS | "
            f"E = {result['energy_eV']:.6f} eV | "
            f"max|F| = {result['max_force_eV_A']:.6f} eV/A"
        )

    except Exception as exc:

        failed += 1

        print("STATUS: FAILED")
        print("Error:", exc)

        failed_file = (
            DFT_FAILED_DIR /
            f"{job_id}_failed.json"
        )

        failure_record = {
            "configuration_id": job_id,
            "molecule_id": row["molecule_id"],
            "molecule_name": row["molecule_name"],
            "family": row["family"],
            "status": "failed",
            "error": str(exc),
            "xyz_file": str(input_file)
        }

        with open(failed_file, "w") as f:
            json.dump(failure_record, f, indent=2)

# ------------------------------------------------------------
# BATCH SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BATCH COMPLETE")
print("=" * 70)
print(f"Batch range : {BATCH_START} - {BATCH_END}")
print(f"Successful  : {successful}")
print(f"Skipped     : {skipped}")
print(f"Failed      : {failed}")
print("=" * 70)

DFT BATCH
Batch range : 157 - 226
Jobs in batch: 70

----------------------------------------------------------------------
JOB 157/563
ID       : M07_C016
Molecule : pyridine
Family   : O_N_aromatic
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------------
JOB 158/563
ID       : M07_C017
Molecule : pyridine
Family   : O_N_aromatic
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------------
JOB 159/563
ID       : M07_C018
Molecule : pyridine
Family   : O_N_aromatic
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------------
JOB 160/563
ID       : M08_C001
Molecule : aniline
Family   : O_N_aromatic
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------------
JOB 161/563
ID       : M08_C002
Molecule : aniline
Family   : O_N_aromatic
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------

In [ ]:
# ============================================================
# CELL 30D — RUN DFT IN BATCHES
# ============================================================

# ============================================================
# ============================================================

import json
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# BATCH SETTINGS
# ------------------------------------------------------------

BATCH_START = 227    # inclusive
BATCH_END   = 337     # inclusive

batch_manifest = manifest.iloc[BATCH_START - 1:BATCH_END].copy()

print("=" * 70)
print("DFT BATCH")
print("=" * 70)
print(f"Batch range : {BATCH_START} - {BATCH_END}")
print(f"Jobs in batch: {len(batch_manifest)}")
print("=" * 70)

successful = 0
skipped = 0
failed = 0

for batch_number, (_, row) in enumerate(
    batch_manifest.iterrows(),
    start=BATCH_START
):

    job_id = row["configuration_id"]

    input_file = Path(row["xyz_path"])
    output_file = DFT_OUTPUT_DIR / f"{job_id}.json"

    print("\n" + "-" * 70)
    print(f"JOB {batch_number}/563")
    print(f"ID       : {job_id}")
    print(f"Molecule : {row['molecule_name']}")
    print(f"Family   : {row['family']}")

    # --------------------------------------------------------
    # CHECK EXISTING OUTPUT
    # --------------------------------------------------------

    if output_file.exists():

        try:
            with open(output_file, "r") as f:
                old_result = json.load(f)

            forces = np.asarray(
                old_result.get("forces_eV_A", []),
                dtype=float
            )

            energy = old_result.get("energy_eV", None)

            valid_existing = (
                old_result.get("status") == "success"
                and energy is not None
                and np.isfinite(float(energy))
                and forces.shape == (int(row["n_atoms"]), 3)
                and np.all(np.isfinite(forces))
                and old_result.get("scf_converged", False)
            )

            if valid_existing:
                print("STATUS: COMPLETED — SKIPPING")
                skipped += 1
                continue

        except Exception:
            pass

    # --------------------------------------------------------
    # RUN DFT
    # --------------------------------------------------------

    print("STATUS: RUNNING")

    try:

        result = run_pyscf_dft_job(
            xyz_file=input_file,
            output_file=output_file,
            method="PBE0",
            dispersion="D3BJ",
            basis="def2-SVP",
            charge=0,
            spin=0
        )

        # Add current manifest metadata
        result.update({
            "configuration_id": job_id,
            "molecule_id": row["molecule_id"],
            "molecule_name": row["molecule_name"],
            "family": row["family"],
        })

        with open(output_file, "w") as f:
            json.dump(result, f, indent=2)

        successful += 1

        print(
            f"STATUS: SUCCESS | "
            f"E = {result['energy_eV']:.6f} eV | "
            f"max|F| = {result['max_force_eV_A']:.6f} eV/A"
        )

    except Exception as exc:

        failed += 1

        print("STATUS: FAILED")
        print("Error:", exc)

        failed_file = (
            DFT_FAILED_DIR /
            f"{job_id}_failed.json"
        )

        failure_record = {
            "configuration_id": job_id,
            "molecule_id": row["molecule_id"],
            "molecule_name": row["molecule_name"],
            "family": row["family"],
            "status": "failed",
            "error": str(exc),
            "xyz_file": str(input_file)
        }

        with open(failed_file, "w") as f:
            json.dump(failure_record, f, indent=2)

# ------------------------------------------------------------
# BATCH SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BATCH COMPLETE")
print("=" * 70)
print(f"Batch range : {BATCH_START} - {BATCH_END}")
print(f"Successful  : {successful}")
print(f"Skipped     : {skipped}")
print(f"Failed      : {failed}")
print("=" * 70)

DFT BATCH
Batch range : 227 - 337
Jobs in batch: 111

----------------------------------------------------------------------
JOB 227/563
ID       : M10_C024
Molecule : BDT
Family   : S_heteroaromatic
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------------
JOB 228/563
ID       : M11_C001
Molecule : benzobithiophene
Family   : S_heteroaromatic
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------------
JOB 229/563
ID       : M11_C002
Molecule : benzobithiophene
Family   : S_heteroaromatic
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------------
JOB 230/563
ID       : M11_C003
Molecule : benzobithiophene
Family   : S_heteroaromatic
STATUS: RUNNING
STATUS: SUCCESS | E = -38379.776671 eV | max|F| = 3.393520 eV/A

----------------------------------------------------------------------
JOB 231/563
ID       : M11_C004
Molecule : benzobithiophene
Family   : S_heteroar

KeyboardInterrupt: 

In [29]:
# ============================================================
# CELL 30D — RUN DFT IN BATCHES
# ============================================================

# ============================================================
# ============================================================

import json
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# BATCH SETTINGS
# ------------------------------------------------------------

BATCH_START = 227    # inclusive
BATCH_END   = 337     # inclusive

batch_manifest = manifest.iloc[BATCH_START - 1:BATCH_END].copy()

print("=" * 70)
print("DFT BATCH")
print("=" * 70)
print(f"Batch range : {BATCH_START} - {BATCH_END}")
print(f"Jobs in batch: {len(batch_manifest)}")
print("=" * 70)

successful = 0
skipped = 0
failed = 0

for batch_number, (_, row) in enumerate(
    batch_manifest.iterrows(),
    start=BATCH_START
):

    job_id = row["configuration_id"]

    input_file = Path(row["xyz_path"])
    output_file = DFT_OUTPUT_DIR / f"{job_id}.json"

    print("\n" + "-" * 70)
    print(f"JOB {batch_number}/563")
    print(f"ID       : {job_id}")
    print(f"Molecule : {row['molecule_name']}")
    print(f"Family   : {row['family']}")

    # --------------------------------------------------------
    # CHECK EXISTING OUTPUT
    # --------------------------------------------------------

    if output_file.exists():

        try:
            with open(output_file, "r") as f:
                old_result = json.load(f)

            forces = np.asarray(
                old_result.get("forces_eV_A", []),
                dtype=float
            )

            energy = old_result.get("energy_eV", None)

            valid_existing = (
                old_result.get("status") == "success"
                and energy is not None
                and np.isfinite(float(energy))
                and forces.shape == (int(row["n_atoms"]), 3)
                and np.all(np.isfinite(forces))
                and old_result.get("scf_converged", False)
            )

            if valid_existing:
                print("STATUS: COMPLETED — SKIPPING")
                skipped += 1
                continue

        except Exception:
            pass

    # --------------------------------------------------------
    # RUN DFT
    # --------------------------------------------------------

    print("STATUS: RUNNING")

    try:

        result = run_pyscf_dft_job(
            xyz_file=input_file,
            output_file=output_file,
            method="PBE0",
            dispersion="D3BJ",
            basis="def2-SVP",
            charge=0,
            spin=0
        )

        # Add current manifest metadata
        result.update({
            "configuration_id": job_id,
            "molecule_id": row["molecule_id"],
            "molecule_name": row["molecule_name"],
            "family": row["family"],
        })

        with open(output_file, "w") as f:
            json.dump(result, f, indent=2)

        successful += 1

        print(
            f"STATUS: SUCCESS | "
            f"E = {result['energy_eV']:.6f} eV | "
            f"max|F| = {result['max_force_eV_A']:.6f} eV/A"
        )

    except Exception as exc:

        failed += 1

        print("STATUS: FAILED")
        print("Error:", exc)

        failed_file = (
            DFT_FAILED_DIR /
            f"{job_id}_failed.json"
        )

        failure_record = {
            "configuration_id": job_id,
            "molecule_id": row["molecule_id"],
            "molecule_name": row["molecule_name"],
            "family": row["family"],
            "status": "failed",
            "error": str(exc),
            "xyz_file": str(input_file)
        }

        with open(failed_file, "w") as f:
            json.dump(failure_record, f, indent=2)

# ------------------------------------------------------------
# BATCH SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BATCH COMPLETE")
print("=" * 70)
print(f"Batch range : {BATCH_START} - {BATCH_END}")
print(f"Successful  : {successful}")
print(f"Skipped     : {skipped}")
print(f"Failed      : {failed}")
print("=" * 70)

DFT BATCH
Batch range : 227 - 337
Jobs in batch: 111

----------------------------------------------------------------------
JOB 227/563
ID       : M10_C024
Molecule : BDT
Family   : S_heteroaromatic
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------------
JOB 228/563
ID       : M11_C001
Molecule : benzobithiophene
Family   : S_heteroaromatic
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------------
JOB 229/563
ID       : M11_C002
Molecule : benzobithiophene
Family   : S_heteroaromatic
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------------
JOB 230/563
ID       : M11_C003
Molecule : benzobithiophene
Family   : S_heteroaromatic
STATUS: COMPLETED — SKIPPING

----------------------------------------------------------------------
JOB 231/563
ID       : M11_C004
Molecule : benzobithiophene
Family   : S_heteroaromatic
STATUS: COMPLETED — SKIPPING

--------------